# Partie II — CNN et classification d'images

**Projet Deep Learning — EMSI Casablanca (2025-2026)**

Notebook autonome pour **Fashion-MNIST** : baseline MLP image, variantes LeNet, CNN amélioré et opérations manuelles.

**Plan**
1. Corrélation croisée et pooling manuels
2. Chargement et visualisation Fashion-MNIST
3. Architectures (MLP, LeNet, ImprovedCNN)
4. Entraînement comparatif
5. Cartes de caractéristiques et export LaTeX

In [ ]:
# Dépendances : pip install torch torchvision scikit-learn matplotlib seaborn pandas numpy tqdm

import os
import random
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

PROJECT_ROOT = Path("..").resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "images"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures" / "cnn"
TABLES_DIR = PROJECT_ROOT / "results" / "tables" / "cnn"
MODELS_DIR = PROJECT_ROOT / "results" / "saved_models" / "cnn"
LOGS_DIR = PROJECT_ROOT / "results" / "logs" / "cnn"
for folder in [DATA_ROOT, FIGURES_DIR, TABLES_DIR, MODELS_DIR, LOGS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
CONFIG = {
    "batch_size": 64,
    "num_workers": 0,
    "train_subset": 12000,
    "val_subset": 2000,
    "test_subset": 3000,
    "mlp_hidden_dims": [256, 128],
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 6,
    "patience": 3,
}
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Périphérique : {device}")

## 1. Opérations CNN manuelles

In [ ]:
def cross_correlation_2d_manual(image, kernel):
    height, width = image.shape
    kh, kw = kernel.shape
    out_h, out_w = height - kh + 1, width - kw + 1
    output = torch.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i : i + kh, j : j + kw]
            output[i, j] = (patch * kernel).sum()
    return output


def max_pool2d_manual(image, kernel_size=2, stride=2):
    height, width = image.shape
    out_h = 1 + (height - kernel_size) // stride
    out_w = 1 + (width - kernel_size) // stride
    output = torch.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i * stride : i * stride + kernel_size, j * stride : j * stride + kernel_size]
            output[i, j] = patch.max()
    return output


def conv_output_size(n, k, p=0, s=1):
    return ((n - k + 2 * p) // s) + 1


def pool_output_size(n, k, s):
    return ((n - k) // s) + 1


sample_image = torch.tensor(
    [[0.0, 1.0, 2.0, 3.0], [1.0, 2.0, 3.0, 4.0], [2.0, 3.0, 4.0, 5.0], [3.0, 4.0, 5.0, 6.0]]
)
kernel = torch.tensor([[1.0, 0.0], [0.0, -1.0]])
manual_corr = cross_correlation_2d_manual(sample_image, kernel)
torch_corr = F.conv2d(sample_image.unsqueeze(0).unsqueeze(0), kernel.unsqueeze(0).unsqueeze(0)).squeeze()
manual_pool = max_pool2d_manual(sample_image)

print("Corrélation manuelle vs PyTorch :", torch.allclose(manual_corr, torch_corr))
print("Taille conv 28×28, k=5, pad=2 :", conv_output_size(28, 5, p=2))
print("Taille conv 28×28, k=5, pad=0 :", conv_output_size(28, 5, p=0))
print("Taille pool 28×28, k=2 :", pool_output_size(28, 2, 2))

formulas_df = pd.DataFrame(
    [
        {"calcul": "Conv 28, k=5, p=2, s=1", "valeur": conv_output_size(28, 5, 2, 1)},
        {"calcul": "Conv 28, k=5, p=0, s=1", "valeur": conv_output_size(28, 5, 0, 1)},
        {"calcul": "Pool 28, k=2, s=2", "valeur": pool_output_size(28, 2, 2)},
    ]
)
formulas_df.to_csv(TABLES_DIR / "cnn_manual_formula_results.csv", index=False)
formulas_df.to_latex(TABLES_DIR / "cnn_manual_formula_results.tex", index=False, float_format="%.4f")
display(formulas_df)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(sample_image.numpy(), cmap="gray")
axes[0].set_title("Entrée 4×4")
axes[1].imshow(manual_corr.numpy(), cmap="viridis")
axes[1].set_title("Corrélation manuelle")
axes[2].imshow(torch_corr.numpy(), cmap="viridis")
axes[2].set_title("Corrélation PyTorch")
for ax in axes:
    ax.axis("off")
plt.suptitle("Comparaison opérations manuelles vs PyTorch")
plt.tight_layout()
fig.savefig(TABLES_DIR / "cnn_manual_ops_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 2. Chargement Fashion-MNIST

In [ ]:
transform = transforms.ToTensor()
train_full = datasets.FashionMNIST(root=str(DATA_ROOT), train=True, download=True, transform=transform)
test_full = datasets.FashionMNIST(root=str(DATA_ROOT), train=False, download=True, transform=transform)

all_indices = np.arange(len(train_full))
train_pool, val_indices = train_test_split(
    all_indices,
    test_size=CONFIG["val_subset"],
    random_state=SEED,
    stratify=train_full.targets.numpy(),
)
train_indices, _ = train_test_split(
    train_pool,
    train_size=CONFIG["train_subset"],
    random_state=SEED,
    stratify=train_full.targets.numpy()[train_pool],
)
test_indices, _ = train_test_split(
    np.arange(len(test_full)),
    train_size=CONFIG["test_subset"],
    random_state=SEED,
    stratify=test_full.targets.numpy(),
)

train_loader = DataLoader(
    Subset(train_full, train_indices),
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
)
val_loader = DataLoader(
    Subset(train_full, val_indices),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)
test_loader = DataLoader(
    Subset(test_full, test_indices),
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)
print(f"Train={len(train_indices)} | Val={len(val_indices)} | Test={len(test_indices)}")

sample_by_class = {}
for image, label in train_full:
    label = int(label)
    if label not in sample_by_class:
        sample_by_class[label] = image
    if len(sample_by_class) == 10:
        break

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for idx, ax in enumerate(axes.flatten()):
    ax.imshow(sample_by_class[idx].squeeze(), cmap="gray")
    ax.set_title(CLASS_NAMES[idx], fontsize=10)
    ax.axis("off")
plt.suptitle("Exemples Fashion-MNIST (1 par classe)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "fashion_mnist_samples.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Architectures

In [ ]:
class ImageMLPBaseline(nn.Module):
    def __init__(self, hidden_dims=None):
        super().__init__()
        hidden_dims = hidden_dims or [256, 128]
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dims[1], 10),
        )

    def forward(self, x):
        return self.network(x)


class LeNetFashion(nn.Module):
    def __init__(self, filters=(16, 32), conv1_padding=2, conv1_stride=1, pool_type="max"):
        super().__init__()
        pool_cls = nn.MaxPool2d if pool_type == "max" else nn.AvgPool2d
        self.features = nn.Sequential(
            nn.Conv2d(1, filters[0], 5, stride=conv1_stride, padding=conv1_padding),
            nn.ReLU(),
            pool_cls(2, 2),
            nn.Conv2d(filters[0], filters[1], 5, padding=2),
            nn.ReLU(),
            pool_cls(2, 2),
        )
        with torch.no_grad():
            flat_dim = int(self.features(torch.zeros(1, 1, 28, 28)).numel())
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 120),
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class ImprovedFashionCNN(nn.Module):
    def __init__(self, filters=(32, 64, 128), use_1x1=True):
        super().__init__()
        layers = [
            nn.Conv2d(1, filters[0], 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(filters[0], filters[1], 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        ]
        if use_1x1:
            layers.extend([nn.Conv2d(filters[1], filters[1], 1), nn.ReLU()])
        layers.extend(
            [
                nn.Conv2d(filters[1], filters[2], 3, padding=1),
                nn.ReLU(),
                nn.MaxPool2d(2),
                nn.AdaptiveAvgPool2d((4, 4)),
            ]
        )
        self.features = nn.Sequential(*layers)
        with torch.no_grad():
            flat_dim = int(self.features(torch.zeros(1, 1, 28, 28)).numel())
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_model(name):
    builders = {
        "mlp_image_baseline": lambda: ImageMLPBaseline(CONFIG["mlp_hidden_dims"]),
        "lenet_default": lambda: LeNetFashion((16, 32), 2, 1, "max"),
        "lenet_padding_zero": lambda: LeNetFashion((16, 32), 0, 1, "max"),
        "lenet_stride_two": lambda: LeNetFashion((16, 32), 2, 2, "max"),
        "lenet_avg_pool": lambda: LeNetFashion((16, 32), 2, 1, "avg"),
        "lenet_more_filters": lambda: LeNetFashion((32, 64), 2, 1, "max"),
        "improved_without_1x1": lambda: ImprovedFashionCNN((32, 64, 128), False),
        "improved_with_1x1": lambda: ImprovedFashionCNN((32, 64, 128), True),
    }
    return builders[name]()

## 4. Entraînement et campagne expérimentale

In [ ]:
def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="weighted", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="weighted", zero_division=0)),
        "f1_score": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }


def plot_confusion_matrix(matrix, labels, title=""):
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        matrix,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Vérité terrain")
    plt.tight_layout()
    return fig


def run_epoch(model, loader, criterion, optimizer, train=True):
    model.train(train)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        if train:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(dim=1) == targets).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


def train_model(model):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    history = {key: [] for key in ["train_loss", "val_loss", "train_accuracy", "val_accuracy"]}
    best_state = deepcopy(model.state_dict())
    best_val_loss = float("inf")
    best_epoch = 0
    patience = 0

    for _ in range(CONFIG["epochs"]):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer, train=True)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer, train=False)
        for key, value in zip(history.keys(), [*train_metrics, *val_metrics]):
            history[key].append(value)
        if val_metrics[0] < best_val_loss:
            best_val_loss = val_metrics[0]
            best_epoch = len(history["val_loss"])
            best_state = deepcopy(model.state_dict())
            patience = 0
        else:
            patience += 1
        if patience >= CONFIG["patience"]:
            break

    model.load_state_dict(best_state)
    return model, history, best_epoch, best_val_loss


@torch.no_grad()
def evaluate_model(model):
    model.eval()
    y_true, y_pred = [], []
    for images, targets in test_loader:
        preds = model(images.to(device)).argmax(dim=1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(targets.numpy().tolist())
    return classification_metrics(y_true, y_pred)


@torch.no_grad()
def extract_feature_maps(model, image_batch):
    model.eval()
    feature_maps = []
    x = image_batch.to(device)
    for layer in model.features:
        x = layer(x)
        if isinstance(layer, nn.Conv2d):
            feature_maps.append(x.cpu())
    return feature_maps

In [ ]:
EXPERIMENTS = [
    "mlp_image_baseline",
    "lenet_default",
    "lenet_padding_zero",
    "lenet_stride_two",
    "lenet_avg_pool",
    "lenet_more_filters",
    "improved_without_1x1",
    "improved_with_1x1",
]

records = []
best_bundle = None
best_name = None

for experiment_name in EXPERIMENTS:
    print(f"\n>>> {experiment_name}")
    model = build_model(experiment_name).to(device)
    model, history, best_epoch, best_val_loss = train_model(model)
    metrics = evaluate_model(model)

    record = {
        "modele": experiment_name,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1_score": metrics["f1_score"],
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
    }
    records.append(record)

    if best_bundle is None or metrics["f1_score"] > best_bundle["metrics"]["f1_score"]:
        best_bundle = {"state_dict": model.state_dict(), "metrics": record}
        best_name = experiment_name

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].legend()
    axes[0].set_title("Pertes")
    axes[1].plot(history["train_accuracy"], label="train")
    axes[1].plot(history["val_accuracy"], label="val")
    axes[1].legend()
    axes[1].set_title("Accuracy")
    plt.suptitle(experiment_name)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f"{experiment_name}_curves.png", dpi=200, bbox_inches="tight")
    plt.show()

    if experiment_name == "lenet_default":
        fig = plot_confusion_matrix(
            metrics["confusion_matrix"],
            CLASS_NAMES,
            title="Matrice de confusion — LeNet (Fashion-MNIST)",
        )
        fig.savefig(
            FIGURES_DIR / f"{experiment_name}_confusion_matrix.png",
            dpi=200,
            bbox_inches="tight",
        )
        plt.show()

    if hasattr(model, "features"):
        batch_images, _ = next(iter(test_loader))
        maps = extract_feature_maps(model, batch_images[:1])
        if maps:
            layer_maps = maps[0][0, :8]
            fig, axes = plt.subplots(2, 4, figsize=(12, 6))
            for idx, ax in enumerate(axes.flatten()):
                if idx < layer_maps.shape[0]:
                    ax.imshow(layer_maps[idx], cmap="magma")
                    ax.set_title(f"Carte {idx + 1}")
                ax.axis("off")
            plt.suptitle(f"{experiment_name} — cartes de caractéristiques")
            fig.savefig(
                FIGURES_DIR / f"{experiment_name}_feature_maps_layer_1.png",
                dpi=200,
                bbox_inches="tight",
            )
            plt.show()

results_df = pd.DataFrame(records).sort_values("f1_score", ascending=False)
results_df.to_csv(TABLES_DIR / "cnn_experiments_comparison.csv", index=False)
results_df.to_latex(TABLES_DIR / "cnn_experiments_comparison.tex", index=False, float_format="%.4f")
display(results_df.style.format({"accuracy": "{:.4f}", "f1_score": "{:.4f}"}))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(results_df["modele"], results_df["accuracy"], color="#d62728")
ax.set_title("Comparaison des accuracy")
ax.set_ylabel("Accuracy")
ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
fig.savefig(FIGURES_DIR / "cnn_accuracy_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

torch.save(best_bundle, MODELS_DIR / "best_cnn_model.pt")
print("Meilleur modèle :", best_name, best_bundle["metrics"])